In [ ]:
pip install google-play-scraper pandas

In [ ]:
from google_play_scraper import Sort, reviews
from datetime import datetime, timedelta, timezone
import pandas as pd
import time

In [ ]:
APP_ID = "com.instagram.android"  
BANYAK_HARI = 60                
LANG = "id"                      
COUNTRY = "id"                  

cutoff_date = datetime.now(timezone.utc) - timedelta(days=BANYAK_HARI)
print(f"Menarik ulasan Play Store dari tanggal: {cutoff_date.strftime('%Y-%m-%d')} sampai hari ini.\n")

all_reviews = []
continuation_token = None
stop_fetching = False

try:
    while not stop_fetching:
        result, continuation_token = reviews(
            APP_ID,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=200,  
            continuation_token=continuation_token
        )

        if not result:
            break

        for item in result:
            review_date = item["at"].replace(tzinfo=timezone.utc)

            if review_date < cutoff_date:
                stop_fetching = True
                break

            all_reviews.append({
                "review_id": item["reviewId"],
                "user_name": item["userName"],
                "rating": item["score"],
                "comment": item["content"],
                "likes": item["thumbsUpCount"],
                "app_version": item["reviewCreatedVersion"],
                "date": review_date.strftime("%Y-%m-%d %H:%M:%S UTC")
            })

        print(f"Berhasil mengumpulkan: {len(all_reviews)} ulasan...")

        if continuation_token is None:
            break

        time.sleep(1)  

    df = pd.DataFrame(all_reviews)
    output_filename = f"ulasan_ig_playstore_2bulan_{LANG}.csv"
    df.to_csv(output_filename, index=False, encoding="utf-8-sig")

    print(f"\nSelesai! Total {len(df)} ulasan berhasil disimpan ke file '{output_filename}'.")

except Exception as e:
    print(f"Terjadi kesalahan: {e}")